# Stage 3, step 2: token-level embeddings for the BioLangFusion encoders

This notebook converts each retained sequence into **a matrix of token features for each of three pretrained encoders**, saves those matrices in reusable cache files, and checks their expected token counts. These cached features are the inputs that [step 3](03_blf_build_check.ipynb) uses to build and train the fusion models.

**Why keep every token?** Stage 1 averaged token features into one vector per sequence. BioLangFusion's fusion methods work with positions within a sequence. Here, each matrix has shape `(number of content tokens, number of features)`: its rows describe token positions, and its columns are learned features. The encoder's final hidden state is its last-layer representation of each token in context. These numerical features are not direct measurements of biological properties.

[Step 1](01_blf_datasets.ipynb) examines the available datasets and the assumptions behind their selection. This notebook reloads the rows through the same shared helper; it does not require an exported dataset from step 1. The reading path is **configure the run → estimate storage → check a few inputs → measure runtime → extract → verify → hand off**.

**Saved execution.** The retained outputs come from the [October 16 run](../../docs/run-records.md#2026-10-16-stage3-token-embeddings-token-level-caches-for-the-four-biolangfusion-benchmarks) on Julian's Mac. They are historical evidence, not a run of this revised notebook. Executing the model-loading cells can download missing weights; executing extraction writes cache files. The current settings schedule all four available benchmarks. This supports Phase A of [Gantt and Details, task 17](../../docs/Research_Project_Plan_Gantt.xlsx); Ab1 remains missing, so coverage is four of the paper's five benchmarks.

---

## Setup

### Imports and shared implementation

NumPy handles arrays and storage dtypes, pandas builds the inspection tables, and PyTorch runs the encoders. Before importing PyTorch, `setdefault` requests CPU fallback for unsupported Apple MPS operations when that setting is not already present. The setting concerns the Apple GPU backend; it does not select the device, and it must be established before PyTorch's first import in a kernel.

The next path block locates the shared package. Its three imports have distinct responsibilities:

- [`datasets.py`](../shared_code/datasets.py): benchmark definitions and the current row-selection rule.
- [`encoders.py`](../shared_code/encoders.py): model/checkpoint definitions, device selection, and loading.
- [`fusion/token_embeddings.py`](../shared_code/fusion/token_embeddings.py), imported as `tokens`: encoder input strings, per-token inference, cache writing, and cache reading.

In [1]:
# Standard library
import datetime
import os
import sys
import time
from pathlib import Path

# Set the Apple GPU fallback option before PyTorch's first import.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import pandas as pd
import torch

In [2]:
# Find Code/ from the working directory or its parents.
# From the repository root, the fallback is its Code/ child directory.
CODE_DIR = Path.cwd() / "Code"
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "shared_code").is_dir():
        CODE_DIR = folder
        break

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Shared dataset rules, encoder loading, and token-cache implementation.
from shared_code import datasets, encoders
from shared_code.fusion import token_embeddings as tokens

### Run settings and selected rows

`TO_RUN` controls which benchmarks the extraction and verification loops process, in the listed order. `BENCHMARK` selects the rows used by the small input check and runtime measurement; changing it alone does not change the extraction list. The current selection starts with CoV-Vac, whose 81-nucleotide inputs make an inexpensive first check.

| Setting | What it controls |
| --- | --- |
| `ENCODER_KEYS` | NT v2 100M multi-species, RNA-FM, and ESM-2 8M, the three encoder choices used for the paper comparison. |
| `CHUNK_SIZE = 256` | Sequences saved per shard. This is a file-writing group, not an inference batch; inference processes one sequence at a time. |
| `DTYPE = np.float16` | Precision of the stored arrays after inference. Two bytes per value halves storage relative to float32, while losing numerical precision. It does not set the model's computation dtype. |
| `CACHE_ROOT` | Parent folder for `<benchmark>/<encoder>/` cache directories. |
| `SAMPLE_N = 24` | Number of initial rows used for the runtime measurement. The earlier input check uses the first eight. |

Smaller shards reduce the arrays accumulated before writing and the work left unsaved after an interruption. Larger shards reduce the number of files. This tradeoff does not determine the model's total GPU-memory requirement.

`Path("embeddings")` is relative to the kernel's **working directory**. With `Code/stage3/` as that directory, it resolves to the intended `Code/stage3/embeddings/`. Finding `shared_code` does not change the working directory. Check the printed absolute path before extraction; the repository ignores `Code/*/embeddings/`, not an arbitrary destination elsewhere. Later Phase B encoder changes require separately identified caches.

In [3]:
# Extraction order: start with the smallest benchmark.
TO_RUN = ["cov_vac_degradation", "fungal_expression", "ecoli_proteins", "mrna_stability"]
BENCHMARK = TO_RUN[0]  # Rows used by the input check and timing measurement.
ENCODER_KEYS = ["ntv2_100m", "rnafm", "esm2_8m"]

CHUNK_SIZE = 256       # Sequences per saved shard, not a model batch size.
DTYPE = np.float16     # Storage precision for returned token features.
CACHE_ROOT = Path("embeddings")  # Relative to the kernel's working directory.
SAMPLE_N = 24          # Initial sequences timed per encoder.

In [4]:
# Resolve the settings and inspect the selected input rows.
DEVICE = encoders.select_device()
ROWS = datasets.blf_benchmark_rows(BENCHMARK)
ENCODERS = [encoders.ENCODERS[k] for k in ENCODER_KEYS]
print(f"device {DEVICE}; cache root {CACHE_ROOT.resolve()}")
for key in TO_RUN:
    print(f"  {datasets.DATASETS[key].label}: {len(datasets.blf_benchmark_rows(key))} sequences")

device cuda; cache root D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage3\embeddings
  CoV-Vac degradation: 2400 sequences
  Fungal expression: 3138 sequences
  E. coli protein expression (low, medium, high): 4450 sequences
  mRNA stability: 23929 sequences


`select_device` prefers CUDA, then Apple MPS, then CPU. `ROWS` contains the selected benchmark after the shared loader drops missing sequences, normalizes letters, and applies the current 999-nucleotide cutoff while preserving file order. [Step 1's preprocessing discussion](01_blf_datasets.ipynb#Checking-BioLangFusion's-preprocessing) explains why this count-matching rule is still an implementation choice rather than a verified reconstruction of the authors' procedure.

The saved output reports an MPS device and a Mac path; those describe the earlier run. A new run prints its own environment. The encoder weights remain fixed during extraction: the code uses evaluation mode and disables gradient recording. There is no random sampling or fitting here, so no seed is needed to select these examples. Reproducibility still depends on the input order, preprocessing, checkpoint/tokenizer versions, software, device, and precision. The registry pins NT's requested revision; RNA-FM and ESM-2 currently have no explicit revision pin.

---

## Estimate the cache size

**Why estimate first?** Retaining every token produces much larger arrays than a single pooled vector. Before loading models, the following loop estimates the numerical-array storage for all four registered benchmarks using sequence lengths, expected token counts, feature widths, and `DTYPE`.

The encoders represent the same input at different positional resolutions. Under the current helpers, an 81-nucleotide CoV-Vac sequence produces:

| Encoder | Content-token rule | Expected matrix shape |
| --- | --- | --- |
| NT v2 100M | Complete six-nucleotide groups, followed by one token per remaining nucleotide: $81 = 13\times6 + 3$ gives 16 tokens. | $16\times512$ |
| RNA-FM | One token per nucleotide. | $81\times640$ |
| ESM-2 8M | One token per translated residue or `X` placeholder in the first-base triplet translation. | $27\times320$ |

These are native encoder resolutions; step 3 will put the streams on a common codon grid. `expected_tokens` calculates NT's count as $\lfloor L/6\rfloor + (L\bmod6)$ for sequence length $L$, not simply $L/6$. For example, 999 nucleotides gives **169** content tokens. Framing tokens added for model input are excluded from cached matrices.

For one encoder and one benchmark, the estimate is:

$$
\text{storage in GB} = \frac{N_{\text{tokens}}\,d\,b}{10^9}.
$$

Here $N_{\text{tokens}}$ is the total expected content-token count across the selected rows, $d$ is the encoder's feature width, and $b$ is bytes per stored value. The widths above are the assumptions used by this estimator; the small model check below reports observed widths. For 2,400 CoV-Vac sequences, RNA-FM alone needs approximately $2{,}400\times81\times640\times2/10^9 = 0.249$ GB of float16 values.

The calculation uses the current `ENCODERS` and `DTYPE` settings. `token_counts` holds one expected count per sequence; `encoder_size_gb` converts their total into storage for one encoder. The loop adds those sizes before rounding the benchmark total. `COST` covers all four registered benchmarks even if `TO_RUN` is shortened.

In [5]:
# Expected feature widths; the smoke test reports the widths returned by the models.
feature_widths = {"ntv2_100m": 512, "rnafm": 640, "esm2_8m": 320}
bytes_per_value = np.dtype(DTYPE).itemsize

cost_rows = []
for benchmark_key in datasets.BLF_BENCHMARKS:
    benchmark_spec = datasets.DATASETS[benchmark_key]
    sequences = datasets.blf_benchmark_rows(benchmark_key)["sequence"].tolist()
    cost_row = {"benchmark": benchmark_spec.label, "sequences": len(sequences)}
    total_size_gb = 0

    for encoder in ENCODERS:
        token_counts = [tokens.expected_tokens(encoder, sequence) for sequence in sequences]
        total_tokens = sum(token_counts)
        feature_width = feature_widths[encoder.key]

        # Token count x features per token x bytes per value, converted to decimal GB.
        encoder_size_gb = total_tokens * feature_width * bytes_per_value / 1e9
        cost_row[f"{encoder.key} GB"] = round(encoder_size_gb, 2)
        total_size_gb += encoder_size_gb

    # Sum the unrounded sizes so display rounding does not change the total.
    cost_row["total GB"] = round(total_size_gb, 2)
    cost_rows.append(cost_row)

COST = pd.DataFrame(cost_rows).set_index("benchmark")
selected_benchmark_label = datasets.DATASETS[BENCHMARK].label
selected_size_gb = COST.loc[selected_benchmark_label, "total GB"]
all_benchmarks_size_gb = COST["total GB"].sum()
print(f"selected: {selected_size_gb} GB   all four: {all_benchmarks_size_gb:.1f} GB")
COST

selected: 0.33 GB   all four: 34.1 GB


,sequences,ntv2_100m GB,rnafm GB,esm2_8m GB,total GB
benchmark,,,,,
Fungal expression,3138,0.35,2.56,0.43,3.33
"E. coli protein expression (low, medium, high)",4450,0.45,3.36,0.56,4.37
mRNA stability,23929,2.71,20.07,3.34,26.12
CoV-Vac degradation,2400,0.04,0.25,0.04,0.33


The saved estimate totals **34.1 GB** after table rounding: about **0.33 GB** for CoV-Vac, **3.33 GB** for fungal expression, **4.37 GB** for E. coli, and **26.12 GB** for mRNA stability. GB means $10^9$ bytes here. These estimates cover feature values; model weights, working memory, archive/metadata overhead, and temporary writing space are separate resource needs.

Float16 is the current storage choice. Frozen weights do not establish that reducing feature precision has no downstream effect; that would need a separate comparison. Loading stored float16 features into float32 later cannot recover the precision already discarded.

---

## Check a few inputs before extraction

The **smoke test** is a small model run intended to expose basic incompatibilities before a larger job. It uses the **first eight rows of `BENCHMARK`**, prepares each encoder's inputs, loads one encoder at a time, and compares returned token counts with `expected_tokens`. This is a deterministic prefix, not a representative sample of every dataset or sequence length.

`inputs_for_rows` converts nucleotide letters to the encoder's alphabet. For ESM-2, it calls `grid_protein`: translate complete triplets from the first supplied base, trim any one or two leftover bases, and replace stop symbols with `X`, an unspecified amino-acid placeholder. This keeps one protein-input position per complete triplet. Stage 1's stop-at-first-stop translation can be shorter, which would change the positional relationship used here.

For example, `AUG GCU UAA` contains three codons. Stop-at-first-stop translation produces `MA`, whereas this notebook's grid-preserving translation produces `MAX`. Here, `X` is an unknown-residue placeholder that preserves the stop codon's position; it does not represent an additional biological amino acid.

This policy is implemented, but its agreement with the authors' handling of fragments and the original biological reading frame remains unresolved.

`token_states` then tokenizes the input, checks its full length against the configured limit, runs inference, and returns the final-layer features after removing start/end/padding tokens. Unknown tokens remain in place. It identifies framing tokens from their actual IDs because the earlier NT run exposed an inconsistent `special_tokens_mask`.

**Earlier diagnosis retained.** The original notebook/helper commentary reported that stop-at-first-stop translation covered all 27 triplets for only 0.7% of CoV-Vac rows and returned an empty string for 6.2%. The following cell does not recompute those percentages. They motivate the alternative input policy but do not establish a biological reading frame; the earlier description of these fragments' construct coordinates also remains to be verified from their source.

In [6]:
sample_sequences = ROWS["sequence"].tolist()[:8]
smoke_results = []

for encoder in ENCODERS:
    input_texts = tokens.inputs_for_rows(encoder, sample_sequences)
    tokenizer, model = encoders.load_encoder(encoder, DEVICE)

    try:
        model.eval()  # Disable training-mode behavior such as dropout.
        features_by_sequence = []
        for input_text in input_texts:
            # token_states disables gradients and returns a CPU array: (tokens, features).
            token_features = tokens.token_states(
                input_text, tokenizer, model, encoder.max_len, DTYPE
            )
            features_by_sequence.append(token_features)
    finally:
        # Drop the model/tokenizer references even if inference raises an error.
        del model, tokenizer

    expected_token_counts = [
        tokens.expected_tokens(encoder, sequence) for sequence in sample_sequences
    ]
    actual_token_counts = [features.shape[0] for features in features_by_sequence]

    if actual_token_counts == expected_token_counts:
        token_count_status = "ok"
    else:
        token_count_status = (
            f"MISMATCH want {expected_token_counts[:3]} got {actual_token_counts[:3]}"
        )

    # Only the protein input must contain one character per complete source triplet.
    protein_grid_status = "ok"
    if encoder.modality == "protein":
        for protein_input, source_sequence in zip(input_texts, sample_sequences):
            expected_residues = len(source_sequence) // 3
            if len(protein_input) != expected_residues:
                protein_grid_status = "MISMATCH"
                break

    first_features = features_by_sequence[0]
    smoke_results.append({
        "encoder": encoder.label,
        "width": first_features.shape[1],
        "dtype": str(first_features.dtype),
        "tokens (first 3)": str(actual_token_counts[:3]),
        "token counts match": token_count_status,
        "protein is one token per codon": protein_grid_status,
    })

pd.DataFrame(smoke_results).set_index("encoder")

W1019 05:43:24.226000 56956 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.weight                 | MISSING    | 
pooler.dense.bias                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,width,dtype,tokens (first 3),token counts match,protein is one token per codon
encoder,,,,,
NT v2 100M multi-species,512,float16,"[16, 16, 16]",ok,ok
RNA-FM,640,float16,"[81, 81, 81]",ok,ok
ESM-2 8M,320,float16,"[27, 27, 27]",ok,ok


The saved CoV-Vac sample reports 16, 81, and 27 tokens, with feature widths 512, 640, and 320, respectively. `token counts match` compares returned lengths with the expected lengths. The protein column checks the prepared protein string's length; its `ok` entries for nucleotide encoders are placeholders because that check does not apply to them.

**Review before continuing.** A length-limit violation raises an exception, but count/grid mismatches are only printed as `MISMATCH`; the extraction cell is not automatically blocked. Inspect the table and resolve any mismatch before extracting. An overflow can reflect changes in rows, tokenizer, encoder, or configured limits. Passing eight short inputs does not test the longest inputs or all biological assumptions.

The saved loading warnings are retained. In particular, the missing pooler parameters concern a pooled output that this helper does not consume: it reads the token states in `out[0]`. This explains that specific warning's relevance; newly missing backbone weights or different warnings require their own investigation.

---

## Measure a small run and project the runtime

The next cell measures inference for the **first `SAMPLE_N` rows of `BENCHMARK`**. Each encoder loads before the timer starts, and a warm-up call absorbs some first-inference overhead. The timed work includes tokenization, inference, transfer back to CPU, and casting returned arrays. It excludes weight loading and cache-file writing.

`RATE` records time per sequence and per 1,000 content tokens. The following cell uses that measured token rate to estimate each benchmark's runtime:

$$
\text{estimated minutes} = \frac{N_{\text{tokens}}}{1{,}000}\times\frac{r}{60},
$$

where $r$ is seconds per 1,000 tokens for that encoder. The benchmark total sums the three encoder estimates. This is a planning approximation: fixed cost per sequence, nonlinear attention cost, sequence lengths, device conditions, disk writing, and cache reuse can all change the realized time.

The helper currently processes one sequence per model call. Batching could improve throughput, but adds padding and memory tradeoffs and would require its own correctness checks. It is a possible future optimization, not part of this readability revision.

In [7]:
timing_sequences = ROWS["sequence"].tolist()[:SAMPLE_N]
timing_rows = []

for encoder in ENCODERS:
    input_texts = tokens.inputs_for_rows(encoder, timing_sequences)
    tokenizer, model = encoders.load_encoder(encoder, DEVICE)

    try:
        model.eval()  # Use evaluation behavior for every timed inference call.

        # Warm up once before timing; model loading is also outside the measured interval.
        tokens.token_states(input_texts[0], tokenizer, model, encoder.max_len, DTYPE)
        start_time = time.perf_counter()
        total_tokens = 0

        for input_text in input_texts:
            token_features = tokens.token_states(
                input_text, tokenizer, model, encoder.max_len, DTYPE
            )
            total_tokens += token_features.shape[0]  # One row per content token.

        elapsed_seconds = time.perf_counter() - start_time
    finally:
        # Release these references on success or failure before loading another encoder.
        del model, tokenizer

    milliseconds_per_sequence = 1000 * elapsed_seconds / len(input_texts)
    seconds_per_1000_tokens = 1000 * elapsed_seconds / total_tokens
    timing_rows.append({
        "encoder": encoder.key,
        "seconds": elapsed_seconds,
        "tokens": total_tokens,
        "ms per sequence": round(milliseconds_per_sequence, 1),
        "seconds per 1k tokens": round(seconds_per_1000_tokens, 3),
    })

RATE = pd.DataFrame(timing_rows).set_index("encoder")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.weight                 | MISSING    | 
pooler.dense.bias                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
projection_rows = []
for benchmark_key in datasets.BLF_BENCHMARKS:
    benchmark_spec = datasets.DATASETS[benchmark_key]
    sequences = datasets.blf_benchmark_rows(benchmark_key)["sequence"].tolist()
    projection_row = {"benchmark": benchmark_spec.label, "sequences": len(sequences)}
    minutes_by_encoder = {}

    for encoder in ENCODERS:
        token_counts = [tokens.expected_tokens(encoder, sequence) for sequence in sequences]
        total_tokens = sum(token_counts)
        seconds_per_1000_tokens = RATE.loc[encoder.key, "seconds per 1k tokens"]

        # Apply the measured rate, then convert seconds to minutes.
        estimated_minutes = total_tokens / 1000 * seconds_per_1000_tokens / 60
        minutes_by_encoder[encoder.key] = estimated_minutes
        projection_row[f"{encoder.key} min"] = round(estimated_minutes)

    # Keep unrounded encoder estimates for the benchmark totals.
    total_minutes = sum(minutes_by_encoder.values())
    projection_row["total minutes"] = round(total_minutes)
    projection_row["total hours"] = round(total_minutes / 60, 1)
    projection_rows.append(projection_row)

PROJECTION = pd.DataFrame(projection_rows).set_index("benchmark")
print(RATE[["ms per sequence", "seconds per 1k tokens"]].to_string())
print(f"\nprojected total for all four benchmarks: {PROJECTION['total hours'].sum():.1f} hours")
PROJECTION

           ms per sequence  seconds per 1k tokens
encoder                                          
ntv2_100m             59.5                  3.722
rnafm                 25.7                  0.317
esm2_8m               17.0                  0.630

projected total for all four benchmarks: 6.5 hours


,sequences,ntv2_100m min,rnafm min,esm2_8m min,total minutes,total hours
benchmark,,,,,,
Fungal expression,3138,21,11,7,38,0.6
"E. coli protein expression (low, medium, high)",4450,27,14,9,50,0.8
mRNA stability,23929,164,83,55,302,5.0
CoV-Vac degradation,2400,2,1,1,4,0.1


The saved output includes loading messages from the measurement, which originally shared a cell with the projection. It reports **12.6, 9.2, and 7.5 milliseconds per sequence** for NT, RNA-FM, and ESM-2 on the short CoV-Vac sample, and a rounded **1.9-hour** projection for all four benchmarks.

The [recorded October 16 extraction](../../docs/run-records.md#2026-10-16-stage3-token-embeddings-token-level-caches-for-the-four-biolangfusion-benchmarks) actually took **52 minutes 41 seconds**. That difference illustrates the limits of projecting all lengths from a short prefix; the direction of the error is not guaranteed on another run or machine. Whole-minute rounding also makes a small nonzero estimate appear as zero in the table.

---

## Extract and save the token features

This loop processes `TO_RUN` in order, finishing the three encoders for one benchmark before moving to the next. `tokens.extract` loads a model only when a shard is missing. It converts each sequence into a token matrix, saves groups of `CHUNK_SIZE` sequences, and prints progress. A completed benchmark can then be used independently of the remaining benchmarks.

A **shard** is one `.npz` file containing concatenated token arrays and a length for each sequence. The lengths locate each sequence within the concatenated array, allowing different sequence lengths without storing padding for every row. The writer first writes a temporary `.partial` file and replaces the final shard filename when the write completes. After an ordinary interruption, completed shards remain and the current unfinished shard may need recomputation.

**What resuming assumes.**

The current implementation writes `meta.json` before the first shard. It records extraction settings—including checkpoint, requested revision, token limit, dtype, sequence count, and chunk size—plus fingerprints of the exact encoder input strings in their order and the `token_embeddings.py` source file. These are the input strings after the encoder-specific transformations. Before resuming, `tokens.extract` compares the recorded fields with the current extraction. If they match, it skips completed shard filenames and computes missing shards; a mismatch stops reuse. Existing shards without the required input metadata cannot be resumed by inventing a new manifest.

These checks protect the recorded input order and settings, but they are not a complete producer record. The manifest does not fully capture resolved model/tokenizer versions, preprocessing code in other modules, or the extraction environment. Resume therefore still requires retaining the producing inputs, their order, model weights, preprocessing settings, and software environment. An existing filename alone cannot establish compatibility or verify the shard's contents. If those identities are uncertain or have changed, review the existing cache and use a separately identified destination for new work. This notebook's final length check cannot replace the identity checks or verify numerical integrity.

For an actual extraction, keep the machine powered and prevent sleep for the intended run. The earlier Mac instructions used `caffeinate -i` in a separate terminal and kept the lid open; that is platform-specific guidance, not a command needed on every host.

In [9]:
# Write or reuse each cache under the compatibility assumptions described above.
for key in TO_RUN:
    rows = datasets.blf_benchmark_rows(key)
    print(f"=== {datasets.DATASETS[key].label}: {len(rows)} sequences "
          f"({datetime.datetime.now():%H:%M:%S}) ===")
    for encoder in ENCODERS:
        cache_dir = CACHE_ROOT / key / encoder.key
        texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
        cache, computed = tokens.extract(
            encoder, texts, cache_dir, DEVICE,
            chunk_size=CHUNK_SIZE, dtype=DTYPE,
        )
        print(f"  {encoder.label}: {'extracted' if computed else 'already complete'}, "
              f"{len(cache)} sequences, {cache.meta['total_tokens']:,} tokens, "
              f"width {cache.width}, {cache.meta['bytes'] / 1e9:.2f} GB")
    print(f"=== done {datetime.datetime.now():%H:%M:%S} ===\n")

=== CoV-Vac degradation: 2400 sequences (05:43:41) ===
    ntv2_100m: shard 1/10 (1/10 this run)
    ntv2_100m: shard 2/10 (2/10 this run)
    ntv2_100m: shard 3/10 (3/10 this run)
    ntv2_100m: shard 4/10 (4/10 this run)
    ntv2_100m: shard 5/10 (5/10 this run)
    ntv2_100m: shard 6/10 (6/10 this run)
    ntv2_100m: shard 7/10 (7/10 this run)
    ntv2_100m: shard 8/10 (8/10 this run)
    ntv2_100m: shard 9/10 (9/10 this run)
    ntv2_100m: shard 10/10 (10/10 this run)
  NT v2 100M multi-species: extracted, 2400 sequences, 38,400 tokens, width 512, 0.04 GB


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.weight                 | MISSING    | 
pooler.dense.bias                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: shard 1/10 (1/10 this run)
    rnafm: shard 2/10 (2/10 this run)
    rnafm: shard 3/10 (3/10 this run)
    rnafm: shard 4/10 (4/10 this run)
    rnafm: shard 5/10 (5/10 this run)
    rnafm: shard 6/10 (6/10 this run)
    rnafm: shard 7/10 (7/10 this run)
    rnafm: shard 8/10 (8/10 this run)
    rnafm: shard 9/10 (9/10 this run)
    rnafm: shard 10/10 (10/10 this run)
  RNA-FM: extracted, 2400 sequences, 194,400 tokens, width 640, 0.25 GB


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.weight                 | MISSING    | 
pooler.dense.bias                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: shard 1/94 (1/94 this run)
    rnafm: shard 2/94 (2/94 this run)
    rnafm: shard 3/94 (3/94 this run)
    rnafm: shard 4/94 (4/94 this run)
    rnafm: shard 5/94 (5/94 this run)
    rnafm: shard 6/94 (6/94 this run)
    rnafm: shard 7/94 (7/94 this run)
    rnafm: shard 8/94 (8/94 this run)
    rnafm: shard 9/94 (9/94 this run)
    rnafm: shard 10/94 (10/94 this run)
    rnafm: shard 11/94 (11/94 this run)
    rnafm: shard 12/94 (12/94 this run)
    rnafm: shard 13/94 (13/94 this run)
    rnafm: shard 14/94 (14/94 this run)
    rnafm: shard 15/94 (15/94 this run)
    rnafm: shard 16/94 (16/94 this run)
    rnafm: shard 17/94 (17/94 this run)
    rnafm: shard 18/94 (18/94 this run)
    rnafm: shard 19/94 (19/94 this run)
    rnafm: shard 20/94 (20/94 this run)
    rnafm: shard 21/94 (21/94 this run)
    rnafm: shard 22/94 (22/94 this run)
    rnafm: shard 23/94 (23/94 this run)
    rnafm: shard 24/94 (24/94 this run)
    rnafm: shard 25/94 (25/94 this run)
    rnafm: shard 2

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    esm2_8m: shard 1/94 (1/94 this run)
    esm2_8m: shard 2/94 (2/94 this run)
    esm2_8m: shard 3/94 (3/94 this run)
    esm2_8m: shard 4/94 (4/94 this run)
    esm2_8m: shard 5/94 (5/94 this run)
    esm2_8m: shard 6/94 (6/94 this run)
    esm2_8m: shard 7/94 (7/94 this run)
    esm2_8m: shard 8/94 (8/94 this run)
    esm2_8m: shard 9/94 (9/94 this run)
    esm2_8m: shard 10/94 (10/94 this run)
    esm2_8m: shard 11/94 (11/94 this run)
    esm2_8m: shard 12/94 (12/94 this run)
    esm2_8m: shard 13/94 (13/94 this run)
    esm2_8m: shard 14/94 (14/94 this run)
    esm2_8m: shard 15/94 (15/94 this run)
    esm2_8m: shard 16/94 (16/94 this run)
    esm2_8m: shard 17/94 (17/94 this run)
    esm2_8m: shard 18/94 (18/94 this run)
    esm2_8m: shard 19/94 (19/94 this run)
    esm2_8m: shard 20/94 (20/94 this run)
    esm2_8m: shard 21/94 (21/94 this run)
    esm2_8m: shard 22/94 (22/94 this run)
    esm2_8m: shard 23/94 (23/94 this run)
    esm2_8m: shard 24/94 (24/94 this run)
    esm2_8

`extracted` means the helper computed at least one missing shard in that call; it does not necessarily mean every shard was newly computed. `already complete` means every expected shard filename was present. The printed sequence/token counts and width describe the resulting cache, and its printed GB value is the estimated feature-array payload.

The saved logs show all four benchmarks finishing. The [run record](../../docs/run-records.md#2026-10-16-stage3-token-embeddings-token-level-caches-for-the-four-biolangfusion-benchmarks) reports **12 caches, 405 shards, and approximately 34.2 GB on disk**, with every shard computed in that run. That recorded disk total is distinct from the rounded 34.1-GB feature estimate above; no new extraction or inspection of those cache files is implied here.

---

## Verify the saved token counts

**Why check each row?** An aggregate byte count could hide a missing or shortened sequence. This cell reloads the current benchmark rows, reads each cache's stored lengths, and compares the complete ordered length vector with `expected_tokens` for those rows. It reports row counts, token totals, width, and the size of files matching `shard_*.npz`.

`all rows match` means **the number of cached rows and their per-row token counts match**. It does not compare sequence contents. Swapping two equal-length sequences could pass this check while attaching the wrong embedding to a target. Width is reported, not asserted against the expected model width; the cell also does not test finite numerical values or quantify float16 precision loss.

If `meta.json` is absent, the report says `not extracted yet`; that label does not prove the cache directory is empty. The current writer saves metadata before creating shards, so an ordinary interruption leaves metadata present. Shards without metadata can belong to a legacy extraction or indicate that metadata was lost; the extractor refuses to resume them without the required identity record. If metadata names a missing shard, `summarize` raises an error: the declared extraction is incomplete. These cases need different recovery decisions.

In [10]:
verification_rows = []
for benchmark_key in TO_RUN:
    benchmark_spec = datasets.DATASETS[benchmark_key]
    benchmark_rows = datasets.blf_benchmark_rows(benchmark_key)

    for encoder in ENCODERS:
        cache_dir = CACHE_ROOT / benchmark_key / encoder.key
        if not (cache_dir / "meta.json").exists():
            verification_rows.append({
                "benchmark": benchmark_spec.label,
                "encoder": encoder.label,
                "all rows match": "not extracted yet",
            })
            continue

        cache = tokens.summarize(cache_dir)
        expected_token_counts = np.array([
            tokens.expected_tokens(encoder, sequence)
            for sequence in benchmark_rows["sequence"]
        ])

        # Compare lengths by row position; matching lengths do not prove sequence identity.
        row_count_matches = len(cache) == len(benchmark_rows)
        token_counts_match = False
        if row_count_matches:
            cached_token_counts = np.asarray(cache.lengths)
            token_counts_match = np.array_equal(cached_token_counts, expected_token_counts)
        row_lengths_match = bool(row_count_matches and token_counts_match)

        # Measure the shard files themselves, including their archive overhead.
        cache_bytes = 0
        for shard_path in cache_dir.glob("shard_*.npz"):
            cache_bytes += shard_path.stat().st_size
        cache_size_gb = round(cache_bytes / 1e9, 2)

        verification_rows.append({
            "benchmark": benchmark_spec.label,
            "encoder": encoder.label,
            "rows": f"{len(cache)} / {len(benchmark_rows)}",
            "all rows match": row_lengths_match,
            "tokens": f"{cache.meta['total_tokens']:,}",
            "width": cache.width,
            "GB on disk": cache_size_gb,
        })

VERIFY = pd.DataFrame(verification_rows).set_index(["benchmark", "encoder"])
# A missing-cache message must count as a failed check, not a truthy string.
every_cache_matches = all(result["all rows match"] is True for result in verification_rows)
print("every row matches:", every_cache_matches)
VERIFY

every row matches: True


rows  \
benchmark                                      encoder                                   
CoV-Vac degradation                            NT v2 100M multi-species    2400 / 2400   
                                               RNA-FM                      2400 / 2400   
                                               ESM-2 8M                    2400 / 2400   
Fungal expression                              NT v2 100M multi-species    3138 / 3138   
                                               RNA-FM                      3138 / 3138   
                                               ESM-2 8M                    3138 / 3138   
E. coli protein expression (low, medium, high) NT v2 100M multi-species    4450 / 4450   
                                               RNA-FM                      4450 / 4450   
                                               ESM-2 8M                    4450 / 4450   
mRNA stability                                 NT v2 100M multi-species  23929 / 23929   
                                               RNA-FM                    23929 / 23929   
                                               ESM-2 8M                  23929 / 23929   

                                                                         all rows match  \
benchmark                                      encoder                                    
CoV-Vac degradation                            NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
Fungal expression                              NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
E. coli protein expression (low, medium, high) NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
mRNA stability                                 NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   

                                                                             tokens  \
benchmark                                      encoder                                
CoV-Vac degradation                            NT v2 100M multi-species      38,400   
                                               RNA-FM                       194,400   
                                               ESM-2 8M                      64,800   
Fungal expression                              NT v2 100M multi-species     337,376   
                                               RNA-FM                     2,000,436   
                                               ESM-2 8M                     666,812   
E. coli protein expression (low, medium, high) NT v2 100M multi-species     442,550   
                                               RNA-FM                     2,622,015   
                                               ESM-2 8M                     874,005   
mRNA stability                                 NT v2 100M multi-species   2,642,703   
                                               RNA-FM                    15,678,663   
                                               ESM-2 8M                   5,226,221   

                                                                         width  \
benchmark                                      encoder                           
CoV-Vac degradation                            NT v2 100M multi-species    512   
                                               RNA-FM                      640   

The saved table reports `True` for all **twelve benchmark–encoder pairs**. This supports the length consistency of those saved results. The run record separately reports an additional October 16 check of finite values, float16 dtype, and widths 512/640/320; those checks are not performed by the cell above and were not repeated during this revision.

Before later results are compared, the cache must also be tied to the dataset contents and row order, input transformations, resolved model/tokenizer versions, and producing software. The current cache manifest does not fully record that identity. The recorded producer used working-tree fixes in addition to its listed base commit, so the base commit alone is not the complete producing-source identity.

For a new authorized run, update the existing [run record](../../docs/run-records.md) with the actual inputs/settings, environment, cache reuse, and checks performed. A green length table should be reported as that specific check, rather than as proof that every prerequisite for model training has been verified.

---

## What step 3 receives

Step 2 supplies a native-resolution token cache for each benchmark and encoder. [`BenchmarkStreams.open`](../shared_code/fusion/blf.py) in step 3 reloads dataset rows and associates the cached matrices with labels and splits **by row position**. That is why preserving the exact dataset identity and order matters even when every matrix has the expected shape.

Step 3 uses these cached features for codon-grid resampling, the three fusion methods, and a shared TextCNN prediction-head architecture. The encoders remain frozen while the downstream fusion and prediction parameters are trained.

The existing smallest-first order remains useful:

| Benchmark | Saved size estimate | Role in the workflow |
| --- | --- | --- |
| CoV-Vac | 0.33 GB | Cheapest extraction check; its short fragments make the protein-input assumption especially visible. |
| Fungal expression | 3.33 GB | Current step 3 build comparison; the paper reports 0.804 for the strongest single encoder and 0.824 for the best main fusion method. Step 1 found no exact sequence overlap across its splits; broader independence is not established by that check. |
| E. coli | 4.37 GB | Classification benchmark, requiring the corresponding downstream loss and metric. |
| mRNA stability | 26.12 GB | Largest extraction; its published-split overlap remains an evaluation limitation. Use measured runtime and available resources when scheduling it. |

### From native tokens to the codon grid

This cache preserves each encoder's own positions. The implemented [`CodonGrid`](../shared_code/fusion/codon_grid.py) in step 3 averages RNA groups of three and resamples DNA features onto the protein-position grid. It provides both transposed-convolution settings discussed in the paper, followed by per-row length handling. Their comparison belongs with the model-building evidence, not with extraction.

An odd number of codons does not produce a “half token” in NT: a partial final six-mer becomes individual nucleotide tokens. Consequently its token positions do not all span the same number of bases. Matching the final array length does not by itself establish biological or positional correspondence. The paper/implementation differences and their consequences remain questions for the build check.

Two useful lessons from the earlier extraction are retained in the [October 16 record](../../docs/run-records.md#2026-10-16-stage3-token-embeddings-token-level-caches-for-the-four-biolangfusion-benchmarks): inspect actual framing-token IDs rather than trusting the inconsistent NT mask, and count remainder nucleotides explicitly. Those corrections explain the current helpers. This notebook supplies reusable representations and evidence about their structure; the subsequent build and evaluation determine what the models can predict.